<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Joint%20Angle%20Trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Joint Angle Trial**
> MARC - Multidisciplinary AI Research Centre | UOP


## **🛠️ Setup OpenDatasetStore**

In [1]:
%%capture
# @title Store Initialization
from google.colab import drive
import pandas as pd
import os
!pip install -q ultralytics opencv-python pandas scipy
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"
store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local",
    entity_id_format="sub_{num:04d}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)

import plotly.io as pio
pio.renderers.default = "colab"


In [9]:
store.summary()
store.list_entries(entry_type="experiments")

📊 Dataset Store Summary
Base Directory : /content/drive/MyDrive/Datasets/Wearable-Biomechanics
Backend        : local
------------------------------
Entities (Total: 13)
  - subjects: 13
------------------------------
Entries (Total: 39)
  - device: 1
  - experiments: 38


,entry_id,entity_id,timestamp,description,raw_csv_path,processed_files,date,time,weight_kg,notes,related_files,processed_metadata,video_sync_side_lag_frames
0,experiments_0001,sub_0001,1778229520,Clean & Jerk,raw_data/experiments/experiments_0001_sub_0001...,{'selected_for_4ang': 'processed_data/experime...,2026-05-08,14:08,20,"5 reps, Sensor detached - left ankle last 3 reps",{'front_video': 'related_files/experiments/exp...,{'selected_for_4ang': {'comment': 'No Issues'}...,89.0
1,experiments_0002,sub_0001,1778230197,Clean & Jerk,raw_data/experiments/experiments_0002_sub_0001...,{'selected_for_4ang': 'processed_data/experime...,2026-05-08,14:20,20,"5 reps, Sensor detached - left ankle in rep 3",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'No Issues'}},NaN
2,experiments_0003,sub_0002,1778666861,Clean & Jerk,raw_data/experiments/experiments_0003_sub_0002...,{},2026-05-13,15:37,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",{'raw_zip': 'related_files/experiments/experim...,{},NaN
3,experiments_0004,sub_0002,1778667119,Clean & Jerk,raw_data/experiments/experiments_0004_sub_0002...,{},2026-05-13,15:42,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",{'raw_zip': 'related_files/experiments/experim...,{},NaN
4,experiments_0005,sub_0003,1778670186,Clean & Jerk,raw_data/experiments/experiments_0005_sub_0003...,{},2026-05-13,16:33,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",{'raw_zip': 'related_files/experiments/experim...,{},NaN
5,experiments_0006,sub_0003,1778671087,Clean & Jerk,raw_data/experiments/experiments_0006_sub_0003...,{},2026-05-13,16:48,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{},NaN
6,experiments_0007,sub_0004,1781068783,Clean & Jerk,raw_data/experiments/experiments_0007_sub_0004...,{'selected_for_4ang': 'processed_data/experime...,2026-06-10,10:49,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'Error in 4t...,NaN
7,experiments_0008,sub_0004,1781069072,Clean & Jerk,raw_data/experiments/experiments_0008_sub_0004...,{'selected_for_4ang': 'processed_data/experime...,2026-06-10,10:54,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'No Issues'}},NaN
8,experiments_0009,sub_0004,1781069235,Clean & Jerk,raw_data/experiments/experiments_0009_sub_0004...,{'selected_for_4ang': 'processed_data/experime...,2026-06-10,10:57,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'Error in IM...,NaN
9,experiments_0010,sub_0004,1781069666,Clean & Jerk,raw_data/experiments/experiments_0010_sub_0004...,{'selected_for_4ang': 'processed_data/experime...,2026-06-10,11:04,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...,{'selected_for_4ang': {'comment': 'Error in 3r...,NaN


# Visual Estimation & Sync Pipeline

In [3]:
# @title Utilities
import cv2
import json
import os
import numpy as np
import pandas as pd
from scipy import signal
import plotly.graph_objects as go
from ultralytics import YOLO

# Initialize YOLO model once
yolo_model = YOLO('yolo11x-pose.pt')

def analyze_keypoints(video_path):
    """
    Analyzes a video file for keypoints, saves the results to a JSON file,
    and returns a pandas DataFrame.
    """
    cap = cv2.VideoCapture(video_path)
    all_frames_data = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = yolo_model(frame, verbose=False)
        frame_data = {"frame": frame_idx, "keypoints": [], "scores": []}

        if len(results) > 0 and results[0].keypoints is not None:
            kpts = results[0].keypoints.xy[0].cpu().numpy().tolist()
            scores = results[0].keypoints.conf[0].cpu().numpy().tolist()

            frame_data["keypoints"] = kpts
            frame_data["scores"] = scores

        all_frames_data.append(frame_data)
        frame_idx += 1

    cap.release()

    # Save to JSON
    output_json_path = video_path.replace('.mp4', '_keypoints.json')
    with open(output_json_path, "w") as f:
        json.dump(all_frames_data, f)

    print(f"Keypoints saved to {output_json_path}")
    return pd.DataFrame(all_frames_data)

def calculate_sync_lag(front_df, side_df, plot=False):
    """
    Calculates the frame lag of the side video compared to the front video.
    Positive lag means side video is behind (lags), negative means front video is behind.
    """
    def get_movement_signal(df):
        coords = []
        for _, row in df.iterrows():
            kpts = row.get('keypoints', [])
            if len(kpts) == 17:
                coords.append(kpts)
            else:
                # Duplicate previous frame to avoid spikes on missing data
                coords.append(coords[-1] if len(coords) > 0 else [[0, 0]] * 17)
        coords = np.array(coords)
        diffs = np.diff(coords, axis=0)
        return np.sum(np.linalg.norm(diffs, axis=2), axis=1)

    sig_front = get_movement_signal(front_df)
    sig_side = get_movement_signal(side_df)

    sig_front_norm = (sig_front - np.mean(sig_front)) / (np.std(sig_front) + 1e-8)
    sig_side_norm = (sig_side - np.mean(sig_side)) / (np.std(sig_side) + 1e-8)

    correlation = signal.correlate(sig_front_norm, sig_side_norm, mode='full')
    lags = signal.correlation_lags(len(sig_front_norm), len(sig_side_norm), mode='full')

    best_lag = lags[np.argmax(correlation)]

    if plot:
        # Original Signals
        fig1 = go.Figure()
        fig1.add_trace(go.Scatter(y=sig_front_norm, mode='lines', name='Front Movement', opacity=0.8))
        fig1.add_trace(go.Scatter(y=sig_side_norm, mode='lines', name='Side Movement', opacity=0.8))
        fig1.update_layout(title="Original Un-Synced Movement Signals", template="plotly_dark", height=400)
        fig1.show()

        # Synced Signals
        fig2 = go.Figure()
        if best_lag > 0:
            x1 = np.arange(len(sig_front))
            x2 = np.arange(len(sig_side)) + best_lag
        else:
            x1 = np.arange(len(sig_front)) - best_lag
            x2 = np.arange(len(sig_side))

        fig2.add_trace(go.Scatter(x=x1, y=sig_front_norm, mode='lines', name='Front Movement (Synced)', opacity=0.8))
        fig2.add_trace(go.Scatter(x=x2, y=sig_side_norm, mode='lines', name='Side Movement (Synced)', opacity=0.8))
        fig2.update_layout(title=f"Aligned Movement Signals (Side Lag: {best_lag} frames)", template="plotly_dark", height=400)
        fig2.show()

    return best_lag

def generate_annotated_video(video_path, keypoints_df):
    """
    Generates an annotated video with skeleton overlays and returns the new file path.
    """
    SKELETON_PAIRS = [
        (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9),
        (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13),
        (13, 15), (12, 14), (14, 16)
    ]
    CONFIDENCE_THRESHOLD = 0.4

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    out_path = video_path.replace('.mp4', '_annotated.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= len(keypoints_df):
            break

        row = keypoints_df.iloc[frame_idx]
        kpts = row.get("keypoints", [])
        scores = row.get("scores", [])

        if len(kpts) == 17:
            for p1, p2 in SKELETON_PAIRS:
                if scores[p1] >= CONFIDENCE_THRESHOLD and scores[p2] >= CONFIDENCE_THRESHOLD:
                    pt1 = (int(kpts[p1][0]), int(kpts[p1][1]))
                    pt2 = (int(kpts[p2][0]), int(kpts[p2][1]))
                    cv2.line(frame, pt1, pt2, (0, 255, 0), 2, lineType=cv2.LINE_AA)

            for i, (x, y) in enumerate(kpts):
                if scores[i] >= CONFIDENCE_THRESHOLD:
                    cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1, lineType=cv2.LINE_AA)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved to {out_path}")
    return out_path

def generate_synced_video(front_video_path, side_video_path, side_lag_frames):
    """
    Generates a synced side-by-side video based on frame lag without annotations.
    """
    cap1 = cv2.VideoCapture(front_video_path)
    cap2 = cv2.VideoCapture(side_video_path)

    if side_lag_frames > 0:
        cap1.set(cv2.CAP_PROP_POS_FRAMES, side_lag_frames)
    elif side_lag_frames < 0:
        cap2.set(cv2.CAP_PROP_POS_FRAMES, -side_lag_frames)

    fps = min(cap1.get(cv2.CAP_PROP_FPS), cap2.get(cv2.CAP_PROP_FPS))
    if fps == 0: fps = 30.0

    ret1, f1 = cap1.read()
    ret2, f2 = cap2.read()

    if ret1 and ret2:
        target_h = f1.shape[0]
        w1 = f1.shape[1]
        w2 = int(f2.shape[1] * (target_h / f2.shape[0]))
        out_w = w1 + w2

        # Reset to synced starting points
        cap1.set(cv2.CAP_PROP_POS_FRAMES, side_lag_frames if side_lag_frames > 0 else 0)
        cap2.set(cv2.CAP_PROP_POS_FRAMES, -side_lag_frames if side_lag_frames < 0 else 0)

        out_path = os.path.join(os.path.dirname(front_video_path), 'synced_side_by_side.mp4')
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(out_path, fourcc, fps, (out_w, target_h))

        while True:
            ret1, frame1 = cap1.read()
            ret2, frame2 = cap2.read()
            if not ret1 or not ret2:
                break

            if frame1.shape[0] != frame2.shape[0]:
                f2_resized = cv2.resize(frame2, (w2, target_h))
            else:
                f2_resized = frame2

            combined = np.hstack((frame1, f2_resized))
            out.write(combined)

        cap1.release()
        cap2.release()
        out.release()
        print(f"Synced video saved to {out_path}")
        return out_path
    else:
        print("Error: Could not read one or both videos.")
        return None

In [6]:
# @title Visual Estimation Pipeline
import os
import glob

def visual_estimation_pipeline(entry_id, store, generate_annotated=True, generate_synced=True):
    """
    Pipeline for visual estimation of a specific experiment.

    Args:
        entry_id (str): The ID of the experiment to process.
        store (OpenDatasetStore): The initialized store object.
        generate_annotated (bool): Whether to generate skeleton-annotated videos.
        generate_synced (bool): Whether to generate a side-by-side synchronized video.

    Returns:
        tuple: (front_df, side_df, side_lag, front_annotated_path, side_annotated_path, synced_video_path)
    """
    entry_type = "experiments"
    labels = ["front_video", "side_video"]

    # Create experiment folder
    exp_folder = f"/content/{entry_id}"
    os.makedirs(exp_folder, exist_ok=True)
    print(f"Setting up directory: {exp_folder}")

    # Download videos
    for label in labels:
        print(f"Downloading {label}...")
        store.download_related_file(
            entry_type=entry_type,
            entry_id=entry_id,
            label=label,
            dest_folder=exp_folder
        )

    # Identify the downloaded files
    all_mp4s = glob.glob(f"{exp_folder}/*.mp4")
    front_video_path = next((f for f in all_mp4s if "front_video" in f), None)
    side_video_path = next((f for f in all_mp4s if "side_video" in f), None)

    if not front_video_path or not side_video_path:
        print("Error: Could not locate both downloaded video files.")
        return None, None, None, None, None, None

    print("\n--- Analyzing Keypoints ---")
    front_df = analyze_keypoints(front_video_path)
    side_df = analyze_keypoints(side_video_path)

    print("\n--- Calculating Sync Lag ---")
    side_lag = calculate_sync_lag(front_df, side_df, plot=True)
    print(f"Calculated side lag: {side_lag} frames")

    front_annotated = None
    side_annotated = None
    synced_video = None

    if generate_annotated:
        print("\n--- Generating Annotated Videos ---")
        front_annotated = generate_annotated_video(front_video_path, front_df)
        side_annotated = generate_annotated_video(side_video_path, side_df)

    if generate_synced:
        print("\n--- Generating Synchronized Video ---")
        synced_video = generate_synced_video(front_video_path, side_video_path, side_lag)

    print("\nPipeline complete!")
    return front_df, side_df, side_lag, front_annotated, side_annotated, synced_video

In [10]:
# @title Save Results
def save_visual_estimation_results(entry_id, store, front_df, side_df, side_lag,
                                   front_annotated_path=None, side_annotated_path=None,
                                   synced_video_path=None):
    """
    Saves the results from the visual estimation pipeline back to the OpenDatasetStore.
    """
    entry_type = "experiments"
    print(f"--- Saving results for {entry_id} to OpenDatasetStore ---")

    # 1. Save Metadata (Lag Frames)
    if side_lag is not None:
        store.edit_entry(
            entry_type=entry_type,
            entry_id=entry_id,
            **{"video_sync_side_lag_frames": int(side_lag)}
        )
        print(f"✓ Saved metadata: video_sync_side_lag_frames = {int(side_lag)}")

    # 2. Save Keypoint DataFrames
    if front_df is not None and not front_df.empty:
        store.add_processed_data(
            entry_type=entry_type,
            entry_id=entry_id,
            tag="front_video_keypoints",
            df=front_df
        )
        print("✓ Saved front_video_keypoints dataframe")

    if side_df is not None and not side_df.empty:
        store.add_processed_data(
            entry_type=entry_type,
            entry_id=entry_id,
            tag="side_video_keypoints",
            df=side_df
        )
        print("✓ Saved side_video_keypoints dataframe")

    # 3. Save Generated Videos
    def save_video(file_path, label):
        if file_path and os.path.exists(file_path):
            try:
                # Assuming the store has an upload_related_file method that accepts file_path
                store.add_related_file(
                    entry_type=entry_type,
                    entry_id=entry_id,
                    label=label,
                    local_path=file_path
                )
                print(f"✓ Saved video file for label: {label}")
            except AttributeError:
                print(f"⚠ Could not automatically save {label}. Store API might require a different method (e.g. upload_related_file_interactive).")
            except Exception as e:
                print(f"⚠ Error saving {label}: {e}")

    save_video(synced_video_path, "synced_video")
    save_video(front_annotated_path, "front_annotated_video")
    save_video(side_annotated_path, "side_annotated_video")

    print("--- Save Complete! ---")


In [7]:
# @title Run Full Pipeline

target_entry_id = "experiments_0001" #@param {type:"string"}

print(f"Starting end-to-end process for {target_entry_id}...")

# 1. Run the pipeline to generate DataFrames and videos
front_df, side_df, side_lag, front_annotated_path, side_annotated_path, synced_video_path = visual_estimation_pipeline(
    entry_id=target_entry_id,
    store=store,
    generate_annotated=True,
    generate_synced=True
)


Starting end-to-end process for experiments_0001...
Setting up directory: /content/experiments_0001
📥 Downloaded to: /content/experiments_0001/experiments_0001_sub_0001_front_video.mp4
📥 Downloaded to: /content/experiments_0001/experiments_0001_sub_0001_side_video.mp4

--- Analyzing Keypoints ---
Keypoints saved to /content/experiments_0001/experiments_0001_sub_0001_front_video_keypoints.json
Keypoints saved to /content/experiments_0001/experiments_0001_sub_0001_side_video_keypoints.json

--- Calculating Sync Lag ---


Calculated side lag: 89 frames

--- Generating Annotated Videos ---
Annotated video saved to /content/experiments_0001/experiments_0001_sub_0001_front_video_annotated.mp4
Annotated video saved to /content/experiments_0001/experiments_0001_sub_0001_side_video_annotated.mp4

--- Generating Synchronized Video ---
Synced video saved to /content/experiments_0001/synced_side_by_side.mp4

Pipeline complete!


In [11]:
# @title Ensure data is valid, then save
if front_df is not None and side_df is not None:
    # 3. Save all results back to the OpenDatasetStore
    save_visual_estimation_results(
        entry_id=target_entry_id,
        store=store,
        front_df=front_df,
        side_df=side_df,
        side_lag=side_lag,
        front_annotated_path=front_annotated_path,
        side_annotated_path=side_annotated_path,
        synced_video_path=synced_video_path
    )
    print(f"\nSuccessfully completed processing and saving for {target_entry_id}!")
else:
    print(f"\nFailed to process {target_entry_id}. Please check the logs above for errors.")

--- Saving results for experiments_0001 to OpenDatasetStore ---
Entry 'experiments_0001' metadata updated.
✓ Saved metadata: video_sync_side_lag_frames = 89
✅ Processed data 'front_video_keypoints' saved and linked to entry 'experiments_0001'.
✓ Saved front_video_keypoints dataframe
✅ Processed data 'side_video_keypoints' saved and linked to entry 'experiments_0001'.
✓ Saved side_video_keypoints dataframe
📎 Attached: experiments_0001_sub_0001_synced_video.mp4
✓ Saved video file for label: synced_video
📎 Attached: experiments_0001_sub_0001_front_annotated_video.mp4
✓ Saved video file for label: front_annotated_video
📎 Attached: experiments_0001_sub_0001_side_annotated_video.mp4
✓ Saved video file for label: side_annotated_video
--- Save Complete! ---

Successfully completed processing and saving for experiments_0001!


# Visual Estimation Testing


In [ ]:
entry_id = "experiments_0010"

In [ ]:
# @title Download Files
import os
import glob

entry_type = "experiments"
labels = ["front_video", "side_video"]

# Create a subfolder for the experiment
exp_folder = f"/content/{entry_id}"
os.makedirs(exp_folder, exist_ok=True)

print(f"Setting up directory: {exp_folder}")

downloaded_videos = []
for label in labels:
    print(f"Downloading {label}...")
    # Download into the designated subfolder
    store.download_related_file(
        entry_type=entry_type,
        entry_id=entry_id,
        label=label,
        dest_folder=exp_folder
    )

# Retrieve the paths of the downloaded mp4 files
downloaded_videos = glob.glob(f"{exp_folder}/*.mp4")
print("\nDownloaded videos:")
for vid in downloaded_videos:
    print(vid)

In [ ]:
# @title Run Visual Estimation
import json
import cv2
import os
from ultralytics import YOLO

# Load the model
model = YOLO('yolo11x-pose.pt')

def extract_keypoints(video_path, output_json_path):
    cap = cv2.VideoCapture(video_path)
    all_frames_data = []
    frame_idx = 0

    print(f"Processing: {os.path.basename(video_path)}")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False)
        frame_data = {"frame": frame_idx, "keypoints": [], "scores": []}

        if len(results) > 0 and results[0].keypoints is not None:
            keypoints = results[0].keypoints.xy[0].cpu().numpy().tolist()
            scores = results[0].keypoints.conf[0].cpu().numpy().tolist()

            frame_data["keypoints"] = keypoints
            frame_data["scores"] = scores

        all_frames_data.append(frame_data)
        frame_idx += 1

    cap.release()

    with open(output_json_path, "w") as f:
        json.dump(all_frames_data, f)
    print(f"Saved {frame_idx} frames to {output_json_path}")

# Run extraction on all downloaded feeds in the folder
json_files = []
for video_path in downloaded_videos:
    base_name = os.path.basename(video_path)
    json_name = base_name.replace('.mp4', '_keypoints.json')
    json_path = os.path.join(exp_folder, json_name)

    extract_keypoints(video_path, json_path)
    json_files.append(json_path)

In [ ]:
# @title Fast Low-Res Visualization
import json
import cv2
import os
import numpy as np
from base64 import b64encode
from IPython.display import HTML, display

SKELETON_PAIRS = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16)
]
CONFIDENCE_THRESHOLD = 0.4
TARGET_WIDTH = 640  # Low resolution target for fast encoding

rendered_videos = []

for video_path in downloaded_videos:
    base_name = os.path.basename(video_path)
    json_path = os.path.join(exp_folder, base_name.replace('.mp4', '_keypoints.json'))
    output_video_path = os.path.join(exp_folder, base_name.replace('.mp4', '_visualized.webm'))

    with open(json_path, "r") as f:
        keypoints_data = json.load(f)

    cap = cv2.VideoCapture(video_path)
    orig_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Calculate scale factors
    scale = TARGET_WIDTH / float(orig_width)
    new_width = int(orig_width * scale)
    new_height = int(orig_height * scale)

    fourcc = cv2.VideoWriter_fourcc(*'VP80')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (new_width, new_height))

    frame_idx = 0
    print(f"\nRendering fast low-res overlay for {base_name}...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= len(keypoints_data):
            break

        # Resize frame immediately to reduce drawing and encoding overhead
        frame = cv2.resize(frame, (new_width, new_height))

        frame_info = keypoints_data[frame_idx]
        kpts = frame_info.get("keypoints", [])
        scores = frame_info.get("scores", [])

        if len(kpts) == 17:
            for p1, p2 in SKELETON_PAIRS:
                if scores[p1] >= CONFIDENCE_THRESHOLD and scores[p2] >= CONFIDENCE_THRESHOLD:
                    # Apply scale to coordinates
                    pt1 = (int(kpts[p1][0] * scale), int(kpts[p1][1] * scale))
                    pt2 = (int(kpts[p2][0] * scale), int(kpts[p2][1] * scale))
                    cv2.line(frame, pt1, pt2, (0, 255, 0), 2, lineType=cv2.LINE_AA)

            for i, (x, y) in enumerate(kpts):
                if scores[i] >= CONFIDENCE_THRESHOLD:
                    cv2.circle(frame, (int(x * scale), int(y * scale)), 3, (0, 0, 255), -1, lineType=cv2.LINE_AA)

        cv2.putText(frame, f"Frame: {frame_idx}", (15, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, lineType=cv2.LINE_AA)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    rendered_videos.append(output_video_path)
    print(f"Visualization complete: {output_video_path}")

html_code = "<div style='display: flex; gap: 20px;'>"

for vid_path in rendered_videos:
    if os.path.exists(vid_path) and os.path.getsize(vid_path) > 0:
        try:
            webm_data = open(vid_path, 'rb').read()
            data_url = "data:video/webm;base64," + b64encode(webm_data).decode()
            html_code += f"""
            <div>
                <h4 style="margin-bottom: 5px;">{os.path.basename(vid_path)}</h4>
                <video width={new_width} controls>
                      <source src="{data_url}" type="video/webm">
                </video>
            </div>
            """
        except Exception as e:
            print(f"Failed to load {vid_path}: {e}")
    else:
        print(f"File missing or empty: {vid_path}")

html_code += "</div>"
display(HTML(html_code))

In [ ]:
import plotly.io as pio
pio.renderers.default = "colab"

In [ ]:
# @title Plot Keypoint Coordinates
import json
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get JSON paths based on the downloaded videos
json_paths = []
for vid in downloaded_videos:
    base = os.path.basename(vid)
    json_path = os.path.join(exp_folder, base.replace('.mp4', '_keypoints.json'))
    if os.path.exists(json_path):
        json_paths.append(json_path)

if len(json_paths) == 2:
    # Create a 2x2 subplot layout
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            f"{os.path.basename(json_paths[0])} - X",
            f"{os.path.basename(json_paths[0])} - Y",
            f"{os.path.basename(json_paths[1])} - X",
            f"{os.path.basename(json_paths[1])} - Y"
        )
    )

    # Loop through the two JSON files
    for i, j_path in enumerate(json_paths):
        with open(j_path, 'r') as f:
            data = json.load(f)

        frames = [d['frame'] for d in data]

        # Plot each of the 17 keypoints
        for kpt_idx in range(17):
            x_coords = []
            y_coords = []
            for d in data:
                # Check if keypoints were detected in this frame
                if len(d.get('keypoints', [])) == 17:
                    x_coords.append(d['keypoints'][kpt_idx][0])
                    y_coords.append(d['keypoints'][kpt_idx][1])
                else:
                    x_coords.append(None)
                    y_coords.append(None)

            # Add X trace (Row i+1, Col 1)
            fig.add_trace(
                go.Scatter(x=frames, y=x_coords, mode='lines', name=f'Keypoint {kpt_idx}',
                           showlegend=(i == 0)),
                row=i+1, col=1
            )

            # Add Y trace (Row i+1, Col 2)
            fig.add_trace(
                go.Scatter(x=frames, y=y_coords, mode='lines', name=f'Keypoint {kpt_idx}',
                           showlegend=False),
                row=i+1, col=2
            )

    # Update layout with the dark theme and sizing
    fig.update_layout(
        height=800,
        width=1200,
        title_text="Keypoint Coordinates over Time (X and Y)",
        template="plotly_dark",
        hovermode="x unified"
    )
    fig.show()
else:
    print(f"Expected 2 JSON files, but found {len(json_paths)}.")

In [ ]:
# @title Synchronize Videos using Keypoint Movement
import json
import os
import numpy as np
from scipy import signal
import plotly.graph_objects as go

if len(json_paths) == 2:
    def get_movement_signal(json_path):
        with open(json_path, 'r') as f:
            data = json.load(f)

        # Extract all keypoints into a numpy array (frames, 17, 2)
        coords = []
        for d in data:
            kpts = d.get('keypoints', [])
            if len(kpts) == 17:
                coords.append(kpts)
            else:
                # If missing, copy the previous frame's keypoints to avoid spikes
                if len(coords) > 0:
                    coords.append(coords[-1])
                else:
                    coords.append([[0, 0]] * 17)
        coords = np.array(coords)

        # Calculate frame-to-frame difference (velocity)
        diffs = np.diff(coords, axis=0)

        # Calculate magnitude of velocity for each keypoint, then sum across all keypoints
        movement = np.sum(np.linalg.norm(diffs, axis=2), axis=1)
        return movement

    # Generate signals for both videos
    sig1 = get_movement_signal(json_paths[0])
    sig2 = get_movement_signal(json_paths[1])

    # Normalize signals for better cross-correlation
    sig1_norm = (sig1 - np.mean(sig1)) / (np.std(sig1) + 1e-8)
    sig2_norm = (sig2 - np.mean(sig2)) / (np.std(sig2) + 1e-8)

    # Perform cross-correlation to find the time delay
    correlation = signal.correlate(sig1_norm, sig2_norm, mode='full')
    lags = signal.correlation_lags(len(sig1_norm), len(sig2_norm), mode='full')

    best_lag = lags[np.argmax(correlation)]

    vid1_name = os.path.basename(json_paths[0]).replace('_keypoints.json', '')
    vid2_name = os.path.basename(json_paths[1]).replace('_keypoints.json', '')

    print(f"--- Synchronization Results ---")
    print(f"Calculated frame offset (lag): {best_lag} frames")
    if best_lag > 0:
        print(f"{vid1_name} is ahead of {vid2_name} by {best_lag} frames.")
    elif best_lag < 0:
        print(f"{vid2_name} is ahead of {vid1_name} by {-best_lag} frames.")
    else:
        print("The videos are perfectly synchronized (0 frame lag).")

    # Plot the aligned signals for visual verification
    fig = go.Figure()

    if best_lag > 0:
        x1 = np.arange(len(sig1))
        x2 = np.arange(len(sig2)) + best_lag
    else:
        x1 = np.arange(len(sig1)) - best_lag
        x2 = np.arange(len(sig2))

    fig.add_trace(go.Scatter(x=x1, y=sig1_norm, mode='lines', name=f'{vid1_name} Movement', opacity=0.8))
    fig.add_trace(go.Scatter(x=x2, y=sig2_norm, mode='lines', name=f'{vid2_name} Movement', opacity=0.8))

    fig.update_layout(
        title=f"Aligned Movement Signals (Offset: {best_lag} frames)",
        xaxis_title="Aligned Frame Index",
        yaxis_title="Normalized Movement Magnitude",
        template="plotly_dark",
        hovermode="x unified",
        height=500
    )
    fig.show()
else:
    print(f"Expected exactly 2 JSON files to synchronize, but found {len(json_paths)}.")

In [ ]:
# @title Visual Verification of Synchronization
import cv2
import numpy as np
import os
from IPython.display import HTML, display
from base64 import b64encode

print("Generating synchronized side-by-side preview...")
vid1_path = downloaded_videos[0]
vid2_path = downloaded_videos[1]

cap1 = cv2.VideoCapture(vid1_path)
cap2 = cv2.VideoCapture(vid2_path)

# Apply the calculated frame offset
if best_lag > 0:
    # Video 1 is ahead, so we skip 'best_lag' frames in Video 1
    cap1.set(cv2.CAP_PROP_POS_FRAMES, best_lag)
elif best_lag < 0:
    # Video 2 is ahead
    cap2.set(cv2.CAP_PROP_POS_FRAMES, -best_lag)

fps = min(cap1.get(cv2.CAP_PROP_FPS), cap2.get(cv2.CAP_PROP_FPS))
if fps == 0: fps = 30.0

target_h = 480
ret1, f1 = cap1.read()
ret2, f2 = cap2.read()

if ret1 and ret2:
    w1 = int(f1.shape[1] * (target_h / f1.shape[0]))
    w2 = int(f2.shape[1] * (target_h / f2.shape[0]))
    out_w = w1 + w2

    # Reset positions to start of sync
    cap1.set(cv2.CAP_PROP_POS_FRAMES, best_lag if best_lag > 0 else 0)
    cap2.set(cv2.CAP_PROP_POS_FRAMES, -best_lag if best_lag < 0 else 0)

    out_path = os.path.join(exp_folder, 'synchronized_playback.webm')
    fourcc = cv2.VideoWriter_fourcc(*'VP80')
    out = cv2.VideoWriter(out_path, fourcc, fps, (out_w, target_h))

    frame_count = 0
    # Render a preview clip (e.g., up to 600 frames / ~20 seconds) to avoid long processing
    MAX_FRAMES = 600

    while frame_count < MAX_FRAMES:
        ret1, frame1 = cap1.read()
        ret2, frame2 = cap2.read()
        if not ret1 or not ret2:
            break

        f1_resized = cv2.resize(frame1, (w1, target_h))
        f2_resized = cv2.resize(frame2, (w2, target_h))

        # Draw original frame numbers for verification
        cv2.putText(f1_resized, f"{vid1_name} Frame: {int(cap1.get(cv2.CAP_PROP_POS_FRAMES))}", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(f2_resized, f"{vid2_name} Frame: {int(cap2.get(cv2.CAP_PROP_POS_FRAMES))}", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        combined = np.hstack((f1_resized, f2_resized))
        out.write(combined)
        frame_count += 1

    cap1.release()
    cap2.release()
    out.release()
    print(f"Saved synchronized video to {out_path} ({frame_count} frames)\n")

    # Display the result
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        webm_data = open(out_path, 'rb').read()
        data_url = "data:video/webm;base64," + b64encode(webm_data).decode()
        html = f'''
        <h4 style="margin-bottom: 5px;">Synchronized Playback Preview (First {MAX_FRAMES} frames)</h4>
        <video width="{out_w}" controls>
              <source src="{data_url}" type="video/webm">
        </video>
        '''
        display(HTML(html))
else:
    print("Failed to read videos.")

In [ ]:
# @title Generate Complete Full-Size Synchronized Video
import cv2
import numpy as np
import os

print("Generating COMPLETE synchronized side-by-side video at FULL resolution...")
print("NOTE: This will take some time. The output file will not be previewed inline to avoid crashing the browser.")

vid1_path = downloaded_videos[0]
vid2_path = downloaded_videos[1]

cap1 = cv2.VideoCapture(vid1_path)
cap2 = cv2.VideoCapture(vid2_path)

# Apply the calculated frame offset
if best_lag > 0:
    cap1.set(cv2.CAP_PROP_POS_FRAMES, best_lag)
elif best_lag < 0:
    cap2.set(cv2.CAP_PROP_POS_FRAMES, -best_lag)

fps = min(cap1.get(cv2.CAP_PROP_FPS), cap2.get(cv2.CAP_PROP_FPS))
if fps == 0: fps = 30.0

ret1, f1 = cap1.read()
ret2, f2 = cap2.read()

if ret1 and ret2:
    # Use the original height of the first video
    target_h = f1.shape[0]
    w1 = f1.shape[1]
    w2 = int(f2.shape[1] * (target_h / f2.shape[0]))
    out_w = w1 + w2

    # Reset positions to start of sync
    cap1.set(cv2.CAP_PROP_POS_FRAMES, best_lag if best_lag > 0 else 0)
    cap2.set(cv2.CAP_PROP_POS_FRAMES, -best_lag if best_lag < 0 else 0)

    out_path = os.path.join(exp_folder, 'synchronized_playback_full.mp4')
    # mp4v is generally faster for high resolution encoding compared to VP8
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (out_w, target_h))

    frame_count = 0
    total_frames_expected = int(min(
        cap1.get(cv2.CAP_PROP_FRAME_COUNT) - (best_lag if best_lag > 0 else 0),
        cap2.get(cv2.CAP_PROP_FRAME_COUNT) - (-best_lag if best_lag < 0 else 0)
    ))

    while True:
        ret1, frame1 = cap1.read()
        ret2, frame2 = cap2.read()
        if not ret1 or not ret2:
            break

        # Resize frame2 only if their heights differ to maintain aspect ratio side-by-side
        if frame1.shape[0] != frame2.shape[0]:
            f2_resized = cv2.resize(frame2, (w2, target_h))
        else:
            f2_resized = frame2

        f1_resized = frame1

        # Draw original frame numbers for verification (scaled up for high-res visibility)
        cv2.putText(f1_resized, f"{vid1_name} Frame: {int(cap1.get(cv2.CAP_PROP_POS_FRAMES))}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 255, 0), 4)
        cv2.putText(f2_resized, f"{vid2_name} Frame: {int(cap2.get(cv2.CAP_PROP_POS_FRAMES))}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 255, 0), 4)

        combined = np.hstack((f1_resized, f2_resized))
        out.write(combined)
        frame_count += 1

        if frame_count % 200 == 0:
            print(f"Processed {frame_count}/{total_frames_expected} frames...")

    cap1.release()
    cap2.release()
    out.release()
    print(f"\nSuccess! Saved full synchronized video to {out_path} ({frame_count} frames)")
    print("Please download the file directly from the Colab file browser panel on the left to view it.")
else:
    print("Failed to read videos.")

In [ ]:
# Replace with your actual file name and desired Drive folder
!cp /content/experiments_0010/synchronized_playback_full_0010.mp4 /content/drive/MyDrive/

In [ ]:
rel_path = store.upload_related_file_interactive(
    entry_type="experiments",
    entry_id=entry_id,
    label="synced_video"
)

In [ ]:
store.edit_entry(
    entry_type="experiments",
    entry_id=entry_id,
    **{"video_sync_side_lag_frames": 7}
)


In [ ]:
# Assuming you have a pandas DataFrame called `filtered_df`
store.add_processed_data(
    entry_type="experiments", # replace with your entry type
    entry_id="exp_001",       # replace with your specific entry ID
    tag="front_video_keypints",           # a short name to identify this specific processed data
    df=keypoints            # your processed pandas DataFrame
)
